# ROGII Exp115 Fast CPU Submission

Fast competition submission notebook for exp115. The model parameters here were selected outside this notebook using only non-heldout OOF rows; this notebook only trains the final residual models and writes `submission.csv`.

In [ ]:
from __future__ import annotations

import gc
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


MODEL_NAMES = [
    'catboost-1',
    'catboost-2',
    'catboost-3',
    'lightgbm-1',
    'lightgbm-2',
    'lightgbm-3',
    'lightgbm-4',
]
ALPHA = 0.001
WELL_WEIGHT = 0.525
ROW_WEIGHT = -0.15
BASE_SMOOTHER = {'name': 'savgol_w571_p2_s0.875', 'kind': 'savgol', 'window': 571, 'polyorder': 2, 'shrink': 0.875}
ROW_SMOOTHER = {'name': 'savgol_w631_p2_s1', 'kind': 'savgol', 'window': 631, 'polyorder': 2, 'shrink': 1.0}
ROW_COLUMNS = [
    'last_known_tvt', 'pf_ancc', 'pf_ancc_std', 'pf_ancc_delta', 'pf_z', 'pf_z_delta', 'pf_vs_z',
    'beam_mean_d', 'beam_std_d', 'sc8_d', 'sc15_d', 'sc25_d', 'sc_cons_d', 'sc_ens_d', 'sc_trust',
    'hyb_d', 'sig_std', 'sig_mean_d', 'tw_range', 'tw_gr_mean', 'grm21', 'grs21', 'grm51', 'grs51',
    'grm101', 'grs101', 'glag1', 'glead1', 'glag5', 'glead5', 'glag15', 'glead15', 'tdsc-15',
    'tdsc-8', 'tdsc0', 'tdsc8', 'tdsc15', 'tdpf-15', 'tdpf-8', 'tdpf0', 'tdpf8', 'tdpf15',
]


def _top_level_entries(path: Path) -> list[str]:
    if not path.exists():
        return []
    return [str(p) for p in sorted(path.iterdir())[:50]]


def _path_sample(paths: list[Path], limit: int = 25) -> list[str]:
    return [str(p) for p in paths[:limit]]


def artifact_train_csv(root: Path) -> Path:
    nested = root / 'data' / 'train.csv'
    flat = root / 'train.csv'
    if nested.exists():
        return nested
    if flat.exists():
        return flat
    return nested


def artifact_oof_paths(root: Path) -> dict[str, Path]:
    paths: dict[str, Path] = {}
    for name in MODEL_NAMES:
        nested = root / 'models' / name / 'oof_preds.pkl'
        flat = root / f'{name}_oof_preds.pkl'
        if nested.exists():
            paths[name] = nested
        elif flat.exists():
            paths[name] = flat
    return paths


def _find_competition_root(input_root: Path) -> Path:
    candidates = [
        input_root / 'rogii-wellbore-geology-prediction',
        input_root / 'competitions' / 'rogii-wellbore-geology-prediction',
    ]
    for path in candidates:
        if (path / 'sample_submission.csv').exists():
            return path
    matches = sorted(input_root.rglob('sample_submission.csv'))
    if matches:
        return matches[0].parent
    raise FileNotFoundError(
        'Could not find competition input containing sample_submission.csv. '
        f'Top-level /kaggle/input entries: {_top_level_entries(input_root)}'
    )


def _find_artifact_root(input_root: Path) -> Path:
    candidates = [
        input_root / 'wellbore-geology-prediction-artifacts',
        input_root / 'datasets' / 'wellbore-geology-prediction-artifacts',
        input_root / 'datasets' / 'ravaghi' / 'wellbore-geology-prediction-artifacts',
    ]
    for path in candidates:
        if artifact_train_csv(path).exists() and artifact_oof_paths(path):
            return path
    for train_csv in sorted(input_root.rglob('train.csv')):
        path = train_csv.parent.parent if train_csv.parent.name == 'data' else train_csv.parent
        if artifact_train_csv(path).exists() and artifact_oof_paths(path):
            return path
    train_hits = sorted(input_root.rglob('train.csv'))
    oof_hits = sorted(input_root.rglob('oof_preds.pkl')) + sorted(input_root.rglob('*_oof_preds.pkl'))
    raise FileNotFoundError(
        'Could not find artifact input containing train.csv plus model OOF pickle files. '
        f'Top-level /kaggle/input entries: {_top_level_entries(input_root)}; '
        f'train.csv hits: {_path_sample(train_hits)}; oof hits: {_path_sample(oof_hits)}'
    )


def resolve_paths() -> tuple[Path, Path, Path]:
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        return _find_competition_root(kaggle_input), _find_artifact_root(kaggle_input), Path('/kaggle/working')

    root = Path.cwd()
    if not (root / 'data').exists() and (root.parent / 'data').exists():
        root = root.parent
    comp_candidates = [
        root / 'data/raw/rogii-wellbore-geology-prediction',
        root / 'data/raw/competitions/rogii-wellbore-geology-prediction',
    ]
    art_candidates = [
        root / 'data/artifacts/wellbore-geology-prediction-artifacts',
        root / 'datasets/wellbore-geology-prediction-artifacts',
        root / 'datasets/ravaghi/wellbore-geology-prediction-artifacts',
    ]
    comp = next((p for p in comp_candidates if (p / 'sample_submission.csv').exists()), comp_candidates[0])
    art = next((p for p in art_candidates if artifact_train_csv(p).exists() and artifact_oof_paths(p)), art_candidates[0])
    return comp, art, Path.cwd()


def load_artifact_frame(art_root: Path) -> tuple[pd.DataFrame, list[str]]:
    train_path = artifact_train_csv(art_root)
    columns = pd.read_csv(train_path, nrows=0).columns.tolist()
    feature_cols = [c for c in columns if c not in ('well', 'id', 'target')]
    dtypes = {c: 'float32' for c in feature_cols + ['target']}
    dtypes.update({'well': 'string', 'id': 'string'})
    return pd.read_csv(train_path, dtype=dtypes), feature_cols


def load_artifact_members(art_root: Path, last_known_tvt: np.ndarray) -> dict[str, np.ndarray]:
    paths = artifact_oof_paths(art_root)
    missing = [name for name in MODEL_NAMES if name not in paths]
    if missing:
        raise FileNotFoundError(f'Missing OOF artifacts for {missing}; available={sorted(paths)}; ART={art_root}')
    members: dict[str, np.ndarray] = {}
    for name in MODEL_NAMES:
        delta = np.asarray(joblib.load(paths[name]), dtype=np.float32)
        members[name] = (last_known_tvt + delta).astype(np.float32)
    return members


def gbr_d4() -> GradientBoostingRegressor:
    return GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.03,
        max_depth=4,
        min_samples_leaf=5,
        random_state=42,
    )


def ridge(alpha: float):
    return make_pipeline(
        StandardScaler(),
        Ridge(alpha=alpha, solver='lsqr', fit_intercept=True),
    )


def build_well_meta(df: pd.DataFrame, feature_cols: list[str]) -> pd.DataFrame:
    core_tokens = ('last_known_tvt', 'pf_', 'beam_', 'sc', 'hyb', 'sig_', 'tw_', 'gr', 'frm_rmse')
    core_cols = [c for c in feature_cols if any(token in c for token in core_tokens)]
    agg = {c: ['mean'] for c in feature_cols}
    for col in core_cols:
        agg[col].append('std')
    meta = df[['well'] + feature_cols].groupby('well', sort=True).agg(agg)
    meta.columns = ['__'.join(col).strip('_') for col in meta.columns.to_flat_index()]
    row_count = df.groupby('well', sort=True).size().rename('row_count').astype('float32')
    return meta.join(row_count).fillna(0.0)


def build_row_features(
    df: pd.DataFrame,
    wells: np.ndarray,
    base: np.ndarray,
    artifact_members: dict[str, np.ndarray],
) -> tuple[np.ndarray, list[str]]:
    selected_cols = [col for col in ROW_COLUMNS if col in df.columns]
    feature_blocks = [df[selected_cols].to_numpy(np.float32, copy=True)]
    feature_names = list(selected_cols)

    member_names = sorted(artifact_members)
    member_stack = np.vstack([artifact_members[name] for name in member_names]).astype(np.float32)
    artifact_std = member_stack.std(axis=0).astype(np.float32)
    artifact_range = (member_stack.max(axis=0) - member_stack.min(axis=0)).astype(np.float32)
    cat_names = [name for name in member_names if name.startswith('catboost')]
    lgb_names = [name for name in member_names if name.startswith('lightgbm')]
    cat_mean = np.mean([artifact_members[name] for name in cat_names], axis=0).astype(np.float32)
    lgb_mean = np.mean([artifact_members[name] for name in lgb_names], axis=0).astype(np.float32)

    row_num = df.groupby('well', sort=False).cumcount().to_numpy(np.float32)
    row_count = df.groupby('well', sort=False)['id'].transform('size').to_numpy(np.float32)
    row_frac = row_num / np.maximum(row_count - 1.0, 1.0)
    last = df['last_known_tvt'].to_numpy(np.float32)
    grouped = pd.DataFrame({'well': wells, 'base': base, 'last': last})
    base_centered = (grouped['base'] - grouped.groupby('well', sort=False)['base'].transform('mean')).to_numpy(np.float32)
    last_centered = (grouped['last'] - grouped.groupby('well', sort=False)['last'].transform('mean')).to_numpy(np.float32)

    derived = np.column_stack([
        base.astype(np.float32),
        artifact_std,
        artifact_range,
        cat_mean,
        lgb_mean,
        (cat_mean - lgb_mean).astype(np.float32),
        row_frac.astype(np.float32),
        np.log1p(row_count).astype(np.float32),
        base_centered,
        last_centered,
    ]).astype(np.float32)
    feature_blocks.append(derived)
    feature_names.extend([
        'artifact_base', 'artifact_std', 'artifact_range', 'catboost_mean', 'lightgbm_mean',
        'catboost_minus_lightgbm', 'row_frac', 'log_row_count', 'base_centered_by_well',
        'last_centered_by_well',
    ])
    X = np.column_stack(feature_blocks).astype(np.float32)
    del member_stack, derived, feature_blocks
    gc.collect()
    return X, feature_names


def _odd_window(requested: int, n: int, polyorder: int) -> int | None:
    if n <= polyorder + 2:
        return None
    window = min(int(requested), n if n % 2 else n - 1)
    if window <= polyorder:
        window = polyorder + 2
        if window % 2 == 0:
            window += 1
    if window > n:
        window = n if n % 2 else n - 1
    if window <= polyorder or window < 3:
        return None
    return window


def smooth_values(values: np.ndarray, candidate: dict[str, object]) -> np.ndarray:
    if str(candidate['kind']) == 'none':
        return values.astype(np.float32, copy=True)
    polyorder = int(candidate['polyorder'])
    window = _odd_window(int(candidate['window']), len(values), polyorder)
    if window is None:
        return values.astype(np.float32, copy=True)
    raw = values.astype(np.float64, copy=True)
    target = savgol_filter(raw, window_length=window, polyorder=polyorder, mode='interp')
    shrink = float(candidate['shrink'])
    return (raw + shrink * (target - raw)).astype(np.float32)


def apply_by_well(pred: np.ndarray, wells: np.ndarray, candidate: dict[str, object]) -> np.ndarray:
    out = pred.astype(np.float32, copy=True)
    frame = pd.DataFrame({'well': wells.astype(str), 'pos': np.arange(len(wells), dtype=np.int64)})
    for _, positions in frame.groupby('well', sort=False)['pos']:
        idx = positions.to_numpy(dtype=np.int64)
        out[idx] = smooth_values(out[idx], candidate)
    return out


t0 = time.time()
COMP, ART, WORKING = resolve_paths()

sample_ids = pd.read_csv(COMP / 'sample_submission.csv', usecols=['id'], dtype={'id': 'string'})
df, feature_cols = load_artifact_frame(ART)
wells = df['well'].astype(str).to_numpy()
ids = df['id'].astype(str).to_numpy()
last = df['last_known_tvt'].to_numpy(np.float32)
y = (last + df['target'].to_numpy(np.float32)).astype(np.float32)

sample_id_values = sample_ids['id'].astype(str).to_numpy()
sample_row_mask = np.isin(ids, sample_id_values)
if int(sample_row_mask.sum()) != len(sample_ids):
    raise ValueError('sample_submission ids do not align one-to-one with artifact train.csv ids')
heldout_wells = np.unique(wells[sample_row_mask])
heldout = np.isin(wells, heldout_wells)
dev = ~heldout

artifact_members = load_artifact_members(ART, last)
base = np.mean(list(artifact_members.values()), axis=0).astype(np.float32)
residual = (y - base).astype(np.float32)

meta = build_well_meta(df, feature_cols)
well_ids = meta.index.astype(str).to_numpy()
Xw = meta.to_numpy(np.float32)
well_to_pos = {well: pos for pos, well in enumerate(well_ids)}
row_well_pos = np.array([well_to_pos[well] for well in wells], dtype=np.int32)
held_well_mask = np.isin(well_ids, heldout_wells)
dev_well_positions = np.flatnonzero(~held_well_mask)
held_well_positions = np.flatnonzero(held_well_mask)
well_target_by_id = (
    pd.DataFrame({'well': wells[dev], 'residual': residual[dev]})
    .groupby('well', sort=True)['residual']
    .mean()
)
well_target = well_target_by_id.loc[well_ids[dev_well_positions]].to_numpy(np.float32)

well_model = gbr_d4()
well_model.fit(Xw[dev_well_positions], well_target)
held_well_correction = well_model.predict(Xw[held_well_positions]).astype(np.float32)
del well_model

gc.collect()
X, row_feature_names = build_row_features(df, wells, base, artifact_members)
del artifact_members

gc.collect()
row_model = ridge(ALPHA)
row_model.fit(X[dev], residual[dev])
held_row_residual = row_model.predict(X[heldout]).astype(np.float32)
del row_model, X

gc.collect()
correction_by_well = np.zeros(len(well_ids), dtype=np.float32)
correction_by_well[held_well_positions] = held_well_correction
held_well_component = correction_by_well[row_well_pos][heldout]
smooth_base = apply_by_well(base[heldout], wells[heldout], BASE_SMOOTHER)
smooth_row = apply_by_well(held_row_residual, wells[heldout], ROW_SMOOTHER)
pred = (smooth_base + WELL_WEIGHT * held_well_component + ROW_WEIGHT * smooth_row).astype(np.float32)

clean_rows = pd.DataFrame({'id': ids[heldout], 'tvt': pred})
submission = sample_ids.astype({'id': 'string'}).merge(clean_rows, on='id', how='left')
if int(submission['tvt'].isna().sum()):
    raise ValueError('missing submission predictions')
if not np.isfinite(submission['tvt'].to_numpy(dtype=np.float64)).all():
    raise ValueError('non-finite submission predictions')
submission.to_csv(WORKING / 'submission.csv', index=False)

print(f'wrote {WORKING / "submission.csv"} rows={len(submission)} elapsed_seconds={time.time() - t0:.1f}')
